# AI & ML Internship — Task 13
## PCA — Dimensionality Reduction

### Objective:
- Apply PCA to reduce feature dimensions
- Track explained variance
- Choose optimal components
- Compare model accuracy before and after PCA
- Visualize PCA projection

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
digits = load_digits()

X = digits.data
y = digits.target

print("Dataset loaded successfully")
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling completed")

In [ ]:
baseline_model = LogisticRegression(max_iter=2000)

baseline_model.fit(X_train_scaled, y_train)

y_pred_base = baseline_model.predict(X_test_scaled)

baseline_acc = accuracy_score(y_test, y_pred_base)

print("Baseline Accuracy (No PCA):", baseline_acc)

In [ ]:
components_list = [2, 10, 30, 50]

pca_results = {}

for n in components_list:
    pca = PCA(n_components=n)

    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    model = LogisticRegression(max_iter=2000)
    model.fit(X_train_pca, y_train)

    acc = accuracy_score(y_test, model.predict(X_test_pca))
    pca_results[n] = acc

    print(f"PCA Components={n} → Accuracy={acc:.4f}")

In [ ]:
pca_full = PCA()
pca_full.fit(X_train_scaled)

cum_var = np.cumsum(pca_full.explained_variance_ratio_)

print("Computed explained variance")

In [ ]:
plt.figure()

plt.plot(cum_var)
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Explained Variance Curve")

plt.grid()

plt.savefig("pca_explained_variance.png", bbox_inches="tight")
plt.show()

In [ ]:
best_k = np.argmax(cum_var >= 0.95) + 1

print("Components needed for 95% variance:", best_k)

In [ ]:
pca_best = PCA(n_components=best_k)

X_train_best = pca_best.fit_transform(X_train_scaled)
X_test_best = pca_best.transform(X_test_scaled)

model_best = LogisticRegression(max_iter=2000)
model_best.fit(X_train_best, y_train)

best_acc = accuracy_score(
    y_test,
    model_best.predict(X_test_best)
)

print("Best PCA Accuracy:", best_acc)

In [ ]:
pca_2d = PCA(n_components=2)

X_2d = pca_2d.fit_transform(X_train_scaled)

plt.figure(figsize=(7,5))

plt.scatter(
    X_2d[:,0],
    X_2d[:,1],
    c=y_train,
    cmap="tab10",
    s=15
)

plt.title("PCA 2D Projection")
plt.xlabel("PC1")
plt.ylabel("PC2")

plt.savefig("pca_2d_scatter.png", bbox_inches="tight")
plt.show()

In [ ]:
reduced_df = pd.DataFrame(X_train_best)
reduced_df["target"] = y_train

reduced_df.to_csv("digits_pca_reduced.csv", index=False)

print("Reduced dataset saved")